In [8]:
import os
import struct
import numpy as np
import tensorflow.compat.v1 as tf
#这版在tf2.x中已经移除 from tensorflow.examples.tutorials.mnist import input_data
#mnist = input_data.read_data_sets('MNIST_data', one_hot=True)


#-------------------------------以下为改写后读取数据集，从本地读---------------------------------------------#

tf.disable_v2_behavior()

# 本地读取 MNIST 原始 idx 文件（不联网下载）
def load_idx_images(file_path):
    with open(file_path, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        if magic != 2051:
            raise ValueError('Invalid image file magic number: {}'.format(magic))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(num, rows * cols).astype(np.float32)

def load_idx_labels(file_path):
    with open(file_path, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        if magic != 2049:
            raise ValueError('Invalid label file magic number: {}'.format(magic))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.astype(np.int64)

def load_mnist_local(root_dir='mnist/MNIST/raw'):
    train_images = load_idx_images(os.path.join(root_dir, 'train-images-idx3-ubyte'))
    train_labels = load_idx_labels(os.path.join(root_dir, 'train-labels-idx1-ubyte'))
    test_images = load_idx_images(os.path.join(root_dir, 't10k-images-idx3-ubyte'))
    test_labels = load_idx_labels(os.path.join(root_dir, 't10k-labels-idx1-ubyte'))

    train_labels_one_hot = np.eye(10, dtype=np.float32)[train_labels]
    test_labels_one_hot = np.eye(10, dtype=np.float32)[test_labels]

    class DataSplit(object):
        def __init__(self, images, labels):
            self.images = images
            self.labels = labels
            self.num_examples = images.shape[0]
            self._index = 0

        def next_batch(self, batch_size):
            if self._index + batch_size > self.num_examples:
                perm = np.random.permutation(self.num_examples)
                self.images = self.images[perm]
                self.labels = self.labels[perm]
                self._index = 0
            start = self._index
            end = start + batch_size
            self._index = end
            return self.images[start:end], self.labels[start:end]

    class MNISTData(object):
        pass

    mnist_data = MNISTData()
    mnist_data.train = DataSplit(train_images, train_labels_one_hot)
    mnist_data.test = DataSplit(test_images, test_labels_one_hot)
    return mnist_data

mnist = load_mnist_local()

#-------------------------------以上为改写后读取数据集，从本地读---------------------------------------------#


#超参数设置
learning_rate = 1e-4
keep_prob_rate = 0.7 # 
max_epoch = 2000

def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1})
    correct_prediction = tf.equal(tf.argmax(y_pre,1), tf.argmax(v_ys,1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1})
    return result

def weight_variable(shape):
    initial = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    # 每一维度  滑动步长全部是 1， padding 方式 选择 same
    # 提示 使用函数  tf.nn.conv2d
    
    return tf.nn.conv2d(x, W, strides=[1,1,1,1], padding='SAME')

def max_pool_2x2(x):
    # 滑动步长 是 2步; 池化窗口的尺度 高和宽度都是2; padding 方式 请选择 same
    # 提示 使用函数  tf.nn.max_pool
    #[batch, height, width, channels] batch=channels=1
    return tf.nn.max_pool(x, ksize=[1,2,2,1], strides=[1,2,2,1], padding='SAME')

# define placeholder for inputs to network
xs = tf.placeholder(tf.float32, [None, 784]) / 255.0
ys = tf.placeholder(tf.float32, [None, 10])

keep_prob = tf.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

#  卷积层 1
## conv1 layer ##

W_conv1 = weight_variable([7, 7, 1, 32])                      # patch 7x7, in size 1, out size 32
b_conv1 = bias_variable([32])                     

h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)                      # 卷积  自己选择 选择激活函数
h_pool1 = max_pool_2x2(h_conv1)                      # 池化               

# 卷积层 2
W_conv2 = weight_variable([5, 5, 32, 64])                       # patch 5x5, in size 32, out size 64
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

#  全连接层 1
## fc1 layer ##
W_fc1 = weight_variable([7*7*64, 1024])
b_fc1 = bias_variable([1024])

h_pool2_flat = tf.reshape(h_pool2, [-1, 7*7*64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob)

# 全连接层 2
## fc2 layer ##
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
logits = tf.matmul(h_fc1_drop, W_fc2) + b_fc2
prediction = tf.nn.softmax(logits)

# 交叉熵函数（数值更稳定）
cross_entropy = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits_v2(logits=logits, labels=ys))
train_step = tf.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

with tf.Session() as sess:
    init = tf.global_variables_initializer()
    sess.run(init)
    
    for i in range(max_epoch):
        batch_xs, batch_ys = mnist.train.next_batch(100)
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob:keep_prob_rate})
        if i % 100 == 0:
            print(compute_accuracy(
                mnist.test.images[:1000], mnist.test.labels[:1000]))


0.121
0.864
0.914
0.939
0.95
0.95
0.952
0.962
0.961
0.962
0.956
0.964
0.963
0.964
0.966
0.959
0.965
0.968
0.967
0.963
